# Actividad 3.2.1 — Aprendizaje automático supervisado
**Clasificador educativo de tumores de mama**. Notebook reproducible, sin uso clínico.

Equipo: Alexandra Bernal Nájera, Luis Antonio Román Rey, Fernando Yair Sanvicente Valencia y Melonie Suárez.

Este cuaderno vuelve a entrenar cuatro algoritmos y comprueba sus resultados.

## 1. Datos y preparación
Breast Cancer Wisconsin (Diagnostic), 569 registros; se usan diez variables medias. **Objetivo:** 1 = maligno, 0 = benigno; se invierte la etiqueta original de scikit-learn.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
b=load_breast_cancer(as_frame=True)
features=list(b.feature_names[:10])
X=b.frame[features]; y=1-b.target
print("Registros:",len(X),"Variables:",len(features))
print("Nulos:",int(X.isna().sum().sum()),"Duplicados:",int(X.duplicated().sum()))
print("Clases (0=benigno, 1=maligno):",y.value_counts().sort_index().to_dict())
display(X.describe().T.round(4))

## 2. Visualización de datos

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots()
y.map({0:"Benigno",1:"Maligno"}).value_counts().plot.bar(ax=ax)
ax.set_title("Balance de clases");ax.set_ylabel("Número de casos")
plt.tight_layout();plt.show()
display(X.corr().round(2))

## 3. Entrenamiento, validación y comparación
Se separa **80 % para entrenamiento y 20 % para prueba**, estratificando por clase. Escalado para KNN y SVM dentro de `Pipeline`. Se optimiza F1 en validación cruzada de cinco pliegues.

In [ ]:
from sklearn.model_selection import StratifiedKFold,GridSearchCV,RepeatedStratifiedKFold,cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix
import joblib,json
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
configs={
"SVM":(Pipeline([("scaler",StandardScaler()),("model",SVC(probability=True,random_state=42))]),{"model__C":[.1,1,10,100],"model__gamma":["scale",.001,.01,.1],"model__kernel":["rbf"]}),
"KNN":(Pipeline([("scaler",StandardScaler()),("model",KNeighborsClassifier())]),{"model__n_neighbors":[3,5,7,9,11],"model__weights":["uniform","distance"]}),
"Random Forest":(RandomForestClassifier(random_state=42,n_jobs=1),{"n_estimators":[100,200],"max_depth":[None,5,10],"min_samples_leaf":[1,2]}),
"Árbol de Decisión":(DecisionTreeClassifier(random_state=42),{"max_depth":[None,3,5,7],"min_samples_leaf":[1,2,4],"criterion":["gini","entropy"]})}
modelos={};filas=[]
for nombre,(modelo,rejilla) in configs.items():
    busq=GridSearchCV(modelo,rejilla,scoring="f1",cv=cv,n_jobs=-1).fit(Xtr,ytr)
    pred=busq.predict(Xte);p=busq.predict_proba(Xte)[:,1]
    modelos[nombre]=busq.best_estimator_
    filas.append({"Modelo":nombre,"F1_CV":busq.best_score_,"Accuracy":accuracy_score(yte,pred),"Precision":precision_score(yte,pred),"Recall":recall_score(yte,pred),"F1":f1_score(yte,pred),"ROC_AUC":roc_auc_score(yte,p),"Parametros":str(busq.best_params_)})
comparacion=pd.DataFrame(filas)
display(comparacion.round(4))

## 4. Validación cruzada repetida y evaluación final

In [ ]:
rep=RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=42)
filas_rep=[]
for nombre,modelo in modelos.items():
    sc=cross_validate(modelo,Xtr,ytr,cv=rep,scoring={"f1":"f1","recall":"recall","precision":"precision"},n_jobs=-1)
    filas_rep.append({"Modelo":nombre,"F1_medio":sc["test_f1"].mean(),"F1_std":sc["test_f1"].std(),"Recall":sc["test_recall"].mean(),"Precision":sc["test_precision"].mean()})
validacion=pd.DataFrame(filas_rep)
display(validacion.round(4))
print("Matriz de confusión SVM (filas reales 0/1, columnas predichas 0/1):")
print(confusion_matrix(yte,modelos["SVM"].predict(Xte)))

## 5. Exportación para la aplicación web
El `.joblib` incluye la normalización y el clasificador en un solo Pipeline. No usar modelos incompatibles con versiones diferentes de scikit-learn.

In [ ]:
base=Path.cwd()
(base/"models").mkdir(exist_ok=True);(base/"data").mkdir(exist_ok=True)
df=X.copy();df["diagnosis"]=y;df.to_csv(base/"data"/"breast_cancer_wisconsin.csv",index=False)
comparacion.to_csv(base/"data"/"comparacion_modelos.csv",index=False)
validacion.to_csv(base/"data"/"validacion_repetida.csv",index=False)
joblib.dump(modelos["SVM"],base/"models"/"modelo.joblib")
r=comparacion.set_index("Modelo").loc["SVM"]
meta={"modelo":"SVM","features":features,"rangos":{f:{"min":float(Xtr[f].min()),"max":float(Xtr[f].max()),"mediana":float(Xtr[f].median())} for f in features},"clases":{"0":"Benigno","1":"Maligno"},"metricas_prueba":{k:float(r[k]) for k in ["Accuracy","Precision","Recall","F1"]}|{"ROC AUC":float(r["ROC_AUC"])}}
(base/"models"/"metadata.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf8")
print("Modelo y metadatos guardados")

**Declaración de IA:** Se utilizó asistencia de IA para revisar y generar borradores de código y documentación. Se requiere revisar, comprender y comprobar el código antes de su entrega.